In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Classifiers and regressors
from sklearn.dummy import DummyRegressor


from sklearn.model_selection import cross_validate

from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.feature_selection import RFECV 
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

#import shap
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_log_error, mean_absolute_percentage_error

from custom_transformers import HostSinceTransformer, BathroomExtractor, UKHostBinaryEncoder, HostResponseOrdinalEncoder

In [3]:
X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")
y_train = pd.read_csv("data/y_train.csv").squeeze()  # Convert to Series
y_test = pd.read_csv("data/y_test.csv").squeeze()

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1465, 28), (629, 28), (1465,), (629,))

In [ ]:
# Load preprocessing
import joblib
preprocessing_pipeline = joblib.load('data/preprocessed/preprocessing_pipeline.pkl')

# Regression with Log(y)

### Baseline Model

In [5]:
y_train_log = np.log(y_train)
y_test_log = np.log(y_test)

pipe_dummy = make_pipeline(preprocessing_pipeline, DummyRegressor())

cv_dummy = cross_validate(pipe_dummy, X_train, y_train_log, cv=5, return_train_score=True)
cv_dummy_score = pd.DataFrame(cv_dummy)
cv_dummy_score

,fit_time,score_time,test_score,train_score
0,0.025243,0.005208,-0.004274,0.0
1,0.008545,0.004374,-0.002520,0.0
2,0.008343,0.004665,-0.015587,0.0
3,0.008318,0.003988,-0.000144,0.0
4,0.007841,0.004045,-0.000553,0.0


### Linear Models

In [6]:
pipe_ridge = make_pipeline(preprocessing_pipeline, Ridge())
cv_ridge = cross_validate(pipe_ridge, X_train, y_train_log, cv=5, return_train_score=True)
cv_ridge_score = pd.DataFrame(cv_ridge)
cv_ridge_score

,fit_time,score_time,test_score,train_score
0,0.012828,0.004769,0.611122,0.654380
1,0.008500,0.004118,0.642066,0.647003
2,0.007992,0.004078,0.538829,0.642046
3,0.008383,0.004241,0.672445,0.640478
4,0.007685,0.003952,0.582789,0.663348


In [7]:
param_grid = {
    "ridge__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}
gs = GridSearchCV(pipe_ridge, param_grid = param_grid, n_jobs=1, return_train_score=True)
gs.fit(X_train, y_train_log)
results = pd.DataFrame(gs.cv_results_)

In [8]:
results_summary = results[["param_ridge__alpha", "mean_test_score", "std_test_score"]]
pd.DataFrame(results_summary)

,param_ridge__alpha,mean_test_score,std_test_score
0,0.001,0.584051,0.090020
1,0.010,0.584404,0.089382
2,0.100,0.587758,0.083324
3,1.000,0.609450,0.046332
4,10.000,0.634998,0.033218
5,100.000,0.628358,0.031125
6,1000.000,0.567497,0.032192


### Tree-based Ensemble Models

In [9]:
pipe_rf = make_pipeline(preprocessing_pipeline, RandomForestRegressor(random_state=123))

pipe_lgbm = make_pipeline(preprocessing_pipeline, LGBMRegressor(random_state=123))

pipe_dtr = make_pipeline(preprocessing_pipeline, DecisionTreeRegressor(random_state=123))

pipe_xgboost = make_pipeline(preprocessing_pipeline, XGBRegressor(random_state=123, verbosity=0))

regressors = {
    "dummy": pipe_dummy,
    "ridge": pipe_ridge,
    "random forest": pipe_rf,
    "LightGBM": pipe_lgbm,
    "decision tree": pipe_dtr,
    "xgboost": pipe_xgboost
}

In [10]:
summary_results = {}

for name, model in regressors.items():
    cv_result = cross_validate(model, X_train, y_train_log, cv=3, return_train_score=True)
    
    summary_results[name] = {
        "fit_time": np.mean(cv_result["fit_time"]),
        "score_time": np.mean(cv_result["score_time"]),
        "mean_test_score": np.mean(cv_result["test_score"]),
        "std_test_score": np.std(cv_result["test_score"]),
        "mean_train_score": np.mean(cv_result["train_score"]),
        "std_train_score": np.std(cv_result["train_score"])
    }
df_summary = pd.DataFrame(summary_results).T

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000861 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1428
[LightGBM] [Info] Number of data points in the train set: 976, number of used features: 25
[LightGBM] [Info] Start training from score 4.481486
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000266 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1415
[LightGBM] [Info] Number of data points in the train set: 977, number of used features: 25
[LightGBM] [Info] Start training from score 4.450614


/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000209 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1425
[LightGBM] [Info] Number of data points in the train set: 977, number of used features: 25
[LightGBM] [Info] Start training from score 4.466375


/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [11]:
df_summary

,fit_time,score_time,mean_test_score,std_test_score,mean_train_score,std_train_score
dummy,0.008107,0.004493,-0.004296,0.003037,0.000000,0.000000
ridge,0.007839,0.004637,0.605521,0.027808,0.650820,0.011603
random forest,0.432484,0.010578,0.754518,0.011562,0.963449,0.002408
LightGBM,0.116247,0.008508,0.771674,0.010422,0.972399,0.001808
decision tree,0.016017,0.005214,0.576455,0.030956,0.989937,0.011632
xgboost,0.096643,0.006653,0.749344,0.009868,0.995609,0.001197


### Feature Selection

In [12]:
pipe_dummy_with_rfecv = make_pipeline(preprocessing_pipeline, RFECV(DecisionTreeRegressor(), cv=3, n_jobs=1), DummyRegressor())

pipe_ridge_with_rfecv = make_pipeline(preprocessing_pipeline, RFECV(DecisionTreeRegressor(), cv=3, n_jobs=1), Ridge())

pipe_rf_with_rfecv = make_pipeline(preprocessing_pipeline, RFECV(DecisionTreeRegressor(), cv=3, n_jobs=1), RandomForestRegressor(random_state=123))

pipe_lgbm_with_rfecv = make_pipeline(preprocessing_pipeline, RFECV(DecisionTreeRegressor(), cv=3, n_jobs=1), LGBMRegressor(random_state=123, verbose=-1))

pipe_dtr_with_rfecv = make_pipeline(preprocessing_pipeline, RFECV(DecisionTreeRegressor(), cv=3, n_jobs=1), DecisionTreeRegressor(random_state=123))

pipe_xgb_with_rfecv = make_pipeline(preprocessing_pipeline, RFECV(DecisionTreeRegressor(), cv=3, n_jobs=1), XGBRegressor(random_state=123, verbosity=0))

regressors_with_rfecv = {
    "dummy": pipe_dummy_with_rfecv,
    "ridge": pipe_ridge_with_rfecv,
    "random forest": pipe_rf_with_rfecv,
    "LightGBM": pipe_lgbm_with_rfecv,
    "decision tree": pipe_dtr_with_rfecv,
    "xgboost": pipe_xgb_with_rfecv
}

In [13]:
REFCV_results = {}

for name, model in regressors_with_rfecv.items():
    model.fit(X_train, y_train_log)
    REFCV_results[name] = model

In [14]:
rfecv_summary = {}
rfecv_scores = {}

for name, model in REFCV_results.items():
    selector = model.named_steps['rfecv']  # Extract RFECV step

    rfecv_scores[name] = {
        "mean_test_score": np.mean(selector.cv_results_["mean_test_score"]),
        "std_test_score": np.mean(selector.cv_results_["std_test_score"]),
    }
    
    rfecv_summary[name] = {
        "Optimal Features": selector.n_features_,
    }

df_rfecv = pd.DataFrame(rfecv_summary).T
df_rfecv_scores = pd.DataFrame(rfecv_scores)
df_rfecv

,Optimal Features
dummy,20
ridge,25
random forest,23
LightGBM,21
decision tree,17
xgboost,25


In [15]:
df_rfecv_scores

,dummy,ridge,random forest,LightGBM,decision tree,xgboost
mean_test_score,0.543735,0.540351,0.547500,0.543680,0.544633,0.541295
std_test_score,0.022813,0.025084,0.025319,0.023831,0.021075,0.023749


In [16]:
df_summary[["mean_test_score", "std_test_score"]].T

,dummy,ridge,random forest,LightGBM,decision tree,xgboost
mean_test_score,-0.004296,0.605521,0.754518,0.771674,0.576455,0.749344
std_test_score,0.003037,0.027808,0.011562,0.010422,0.030956,0.009868


### Hyperparameter Optimization for Best Model

### LightGB

In [17]:
param_random_lightGB = {
    "lgbmregressor__n_estimators": np.arange(100, 600, 50),  # Number of boosting rounds
    "lgbmregressor__learning_rate": np.logspace(-3, -1, 5), # Step size 
    "lgbmregressor__max_depth": np.arange(3, 10, 2), # Tree depth 
    "lgbmregressor__num_leaves": np.arange(10, 50, 10)  # Number of leaves
}

rs_lightGBM = RandomizedSearchCV(pipe_lgbm, param_random_lightGB,  n_iter=100, cv=4, n_jobs=1, random_state=123)
rs_lightGBM.fit(X_train, y_train_log)

/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor wa

,estimator,Pipeline(step..._state=123))])
,param_distributions,"{'lgbmregressor__learning_rate': array([0.001 ..., 0.1 ]), 'lgbmregressor__max_depth': array([3, 5, 7, 9]), 'lgbmregressor__n_estimators': array([100, 1...50, 500, 550]), 'lgbmregressor__num_leaves': array([10, 20, 30, 40])}"
,n_iter,100
,scoring,None
,n_jobs,1
,refit,True
,cv,4
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,123
,error_score,nan


In [18]:
# Get the best score and hyperparameter values
rs_lightGBM.best_score_, rs_lightGBM.best_params_

(0.7789613896874225,
 {'lgbmregressor__num_leaves': 10,
  'lgbmregressor__n_estimators': 550,
  'lgbmregressor__max_depth': 5,
  'lgbmregressor__learning_rate': 0.03162277660168379})

### Stacking

In [19]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV


estimators = [
    ('rf', RandomForestRegressor(n_estimators=100, random_state=123)),
    ('lgb', LGBMRegressor(random_state=123)),
    ('xgb', XGBRegressor(random_state=123, verbosity=0))
]

stacking_reg = StackingRegressor(
    estimators=estimators,
    final_estimator=RidgeCV(),
    passthrough=True,
    cv=5,
    n_jobs=1
)

stacking_model = make_pipeline(
    preprocessing_pipeline,
    stacking_reg
)

# Example param grid for RidgeCV (final_estimator) and base estimators
param_dist = {
    'stackingregressor__final_estimator__alphas': [np.logspace(-3, 3, 7)],
    'stackingregressor__rf__n_estimators': [50, 100, 200],
    'stackingregressor__lgb__num_leaves': [15, 31, 63],
    'stackingregressor__xgb__max_depth': [3, 5, 7]
}

rs_stacking_model = RandomizedSearchCV(
    stacking_model,
    param_distributions=param_dist,
    n_iter=10,
    cv=4,
    n_jobs=1,
    random_state=123
)
rs_stacking_model.fit(X_train, y_train_log)

rs_stacking_model.best_score_, rs_stacking_model.best_params_

/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor wa

(0.7586148791932221,
 {'stackingregressor__xgb__max_depth': 3,
  'stackingregressor__rf__n_estimators': 100,
  'stackingregressor__lgb__num_leaves': 63,
  'stackingregressor__final_estimator__alphas': array([1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02, 1.e+03])})

### Scoring

In [20]:
# Best model so far
best_model_regression = rs_lightGBM.best_estimator_
y_pred_log = best_model_regression.predict(X_test)

/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [21]:
r2_log = r2_score(y_test_log, y_pred_log)
mae_log = mean_absolute_error(y_test_log, y_pred_log)
rmsle_log = mean_squared_log_error(y_test_log, y_pred_log)
mape_log = mean_absolute_percentage_error(y_test_log, y_pred_log)

print(f"R² on log scale: {r2_log:.3f}")
print(f"MAE on log scale: ${mae_log:.2f}")
print(f"RMSLE on log scale: {rmsle_log:.4f}")
print(f"MAPE on log scale: ${mape_log:.2f}")


R² on log scale: 0.727
MAE on log scale: $0.22
RMSLE on log scale: 0.0031
MAPE on log scale: $0.05


A MAE of $0.22 on the log scale means that, on average, the predicted log(price) differs from the actual log(price) by 0.22 units. This translates to a typical multiplicative error of about ±25% in the original price (since exp(0.22) ≈ 1.25).

**Explanation of all regression scores on the log scale and whether the results are good:**

- **R² on log scale (0.727):**  
    The model explains 72.7% of the variance in log-transformed prices. In other words, if the true log(price) varies a lot, the model can capture most of those variations. Hence, this is a good score.

- **MAE on log scale ($0.22):**  
    The average absolute difference between predicted and actual log(price) is 0.22. In terms of the original price, predictions are typically within +-25% of the true value. Say, if the true price is $100, the predicted price will usually be between $80 and $125. Hence, for pricing tasks, a +-25% error is reasonable.  

- **RMSLE on log scale (0.0031):**  
    The root mean squared logarithmic error measures the average squared difference between predicted and actual log(price). Thus, lower values indicate better performance. For example: Most predictions are close to the actual value, with very few large mistakes.
    Hence 0.0031 is a good score and is very low, showing the model rarely makes large errors.
- **MAPE on log scale (0.05):**  
    The mean absolute percentage error is 0.05, meaning the average proportional error in log(price) predictions is 5%. If the true price is $200, the average error is about $10. So, a 5% average error is excellent for regression tasks

**Note:**  
All scores above are calculated on log-transformed prices, so errors represent proportional differences rather than absolute dollar amounts. To interpret in terms of actual price, exponentiate the error values. For example, a log MAE of 0.22 means the predicted price is typically between 80% and 125% of the actual price. Which can be found below

In [22]:
# Convert back to original scale
y_pred_real = np.exp(y_pred_log)
y_test_real = np.exp(y_test_log)  # If y_test was also log-transformed

In [23]:
r2 = r2_score(y_test_real, y_pred_real)
mae = mean_absolute_error(y_test_real, y_pred_real)
rmsle = mean_squared_log_error(y_test_real, y_pred_real)
mape = mean_absolute_percentage_error(y_test_real, y_pred_real)

print(f"RMSLE: {rmsle:.4f}")
print(f"R² on original scale: {r2:.3f}")
print(f"MAE on original price scale: ${mae:.2f}")
print(f"MAPE on original scale: {mape:.2f}")


RMSLE: 0.0880
R² on original scale: 0.652
MAE on original price scale: $22.43
MAPE on original scale: 0.22


# Interquartile Binning

Let's use interquartile binning instead of equal width binning. This way we can avoid issues like class imbalance, which can greatly affect many models' predictability.

In [24]:
# Bin into 5 equal-width intervals
y_train_bin = pd.qcut(y_train, q=8, labels=False)
y_test_bin = pd.qcut(y_test, q=8, labels=False)

### Baseline

In [25]:
from sklearn.dummy import DummyClassifier

pipe_dummyc = make_pipeline(preprocessing_pipeline, DummyClassifier(strategy='most_frequent'))
cv_dummy_classification = cross_validate(pipe_dummyc, X_train, y_train_bin, cv=5, return_train_score=True)
cv_dummy_classification_score = pd.DataFrame(cv_dummy_classification)
cv_dummy_classification_score

,fit_time,score_time,test_score,train_score
0,0.009571,0.005699,0.129693,0.129693
1,0.009165,0.005195,0.129693,0.129693
2,0.008890,0.004616,0.129693,0.129693
3,0.008431,0.004512,0.129693,0.129693
4,0.008307,0.004899,0.129693,0.129693


### Logistic Regression

In [26]:
from sklearn.linear_model import LogisticRegression

pipe_logistic_regression = make_pipeline(preprocessing_pipeline, LogisticRegression(max_iter=50000, solver="lbfgs", penalty='l2', random_state=123))
cv_logistic_regression = cross_validate(pipe_logistic_regression, X_train, y_train_bin, cv=5, return_train_score=True)
cv_logistic_regression_score = pd.DataFrame(cv_logistic_regression)
cv_logistic_regression_score

/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 14121 iteration(s) (status=1):
STOP: TOTAL NO. OF F,G EVALUATIONS EXCEEDS LIMIT

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 14152 iteration(s) (status=1):
STOP: TOTAL NO. OF F,G EVALUATIONS EXCEEDS LIMIT

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linea

,fit_time,score_time,test_score,train_score
0,3.249331,0.006306,0.358362,0.431741
1,3.223988,0.006003,0.361775,0.436007
2,3.241300,0.006383,0.402730,0.417235
3,3.156801,0.005917,0.358362,0.418942
4,3.206264,0.006292,0.331058,0.419795


Here, both the train and test scores are low, which suggests that the logistic regression model is underfitting. To address this, I relaxed the regularization by testing much larger values of C and also increased the max_iter parameter to help the model converge.

In [27]:
param_grid = {
    "logisticregression__C": [10, 100, 1000, 10000, 100000, 1000000, 10000000, 100000000, 1000000000, 10000000000]
}
gs = GridSearchCV(pipe_logistic_regression, param_grid = param_grid, n_jobs=1, return_train_score=True)
gs.fit(X_train, y_train_bin)
results = pd.DataFrame(gs.cv_results_)
results_summary = results[["param_logisticregression__C", "mean_test_score", "std_test_score", "mean_train_score", "std_train_score"]]
pd.DataFrame(results_summary)


/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 14152 iteration(s) (status=1):
STOP: TOTAL NO. OF F,G EVALUATIONS EXCEEDS LIMIT

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 14129 iteration(s) (status=1):
STOP: TOTAL NO. OF F,G EVALUATIONS EXCEEDS LIMIT

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linea

,param_logisticregression__C,mean_test_score,std_test_score,mean_train_score,std_train_score
0,10,0.367235,0.020318,0.427645,0.008138
1,100,0.365188,0.019307,0.426280,0.012442
2,1000,0.365870,0.024931,0.425427,0.010828
3,10000,0.367918,0.026472,0.427816,0.008644
4,100000,0.371331,0.017721,0.427986,0.006775
5,1000000,0.360410,0.021215,0.427133,0.011879
6,10000000,0.362457,0.024172,0.426280,0.014536
7,100000000,0.366553,0.022598,0.428157,0.011889
8,1000000000,0.365188,0.028309,0.424061,0.012435
9,10000000000,0.365188,0.021477,0.428157,0.009689


### SVC Model

In [28]:
from sklearn.svm import SVC


pipe_svc = make_pipeline(preprocessing_pipeline, SVC(kernel='rbf', probability=True, random_state=123))
cv_svc = cross_validate(pipe_svc, X_train, y_train_bin, cv=5, return_train_score=True)
cv_svc_score = pd.DataFrame(cv_svc)
cv_svc_score

,fit_time,score_time,test_score,train_score
0,0.360666,0.026853,0.136519,0.159556
1,0.353878,0.026193,0.143345,0.159556
2,0.353879,0.026241,0.170648,0.148464
3,0.350838,0.026371,0.119454,0.162969
4,0.354613,0.026214,0.160410,0.152730


### Tree-based Ensemble Models

In [34]:
### Tree-based Ensemble Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier


pipe_rfc = make_pipeline(
    preprocessing_pipeline,
    RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        min_samples_leaf=10,
        max_features='sqrt',
        random_state=123
    )
)

pipe_lgbmc = make_pipeline(preprocessing_pipeline, XGBClassifier(random_state=123))

pipe_dtrc = make_pipeline(
    preprocessing_pipeline,
    DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=10,
        max_features='sqrt',
        random_state=123
    )
)

pipe_xgboostc = make_pipeline(
    preprocessing_pipeline,
    XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='mlogloss',
        verbosity=0,
        random_state=123
    )
)


classifiers = {
    "dummy": pipe_dummyc,
    "random forest": pipe_rfc,
    "LightGBM": pipe_lgbmc,
    "decision tree": pipe_dtrc,
    "xgboost": pipe_xgboostc
}

# Use multiple scorers
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc_ovr']

summary_results = {}

for name, model in classifiers.items():
    cv_result = cross_validate(
        model,
        X_train,
        y_train_bin,
        scoring=scoring,
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=123),
        return_train_score=True
    )

    summary_results[name] = {
        "fit_time": np.mean(cv_result["fit_time"]),
        "score_time": np.mean(cv_result["score_time"]),
        "mean_train_accuracy": np.mean(cv_result["train_accuracy"]),
        "mean_test_accuracy": np.mean(cv_result["test_accuracy"]),
        "mean_precision": np.mean(cv_result["test_precision_macro"]),
        "mean_recall": np.mean(cv_result["test_recall_macro"]),
        "mean_f1": np.mean(cv_result["test_f1_macro"]),
        "mean_auc": np.mean(cv_result["test_roc_auc_ovr"]),
    }

df_summary = pd.DataFrame(summary_results).T
df_summary

/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/portfolio/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

,fit_time,score_time,mean_train_accuracy,mean_test_accuracy,mean_precision,mean_recall,mean_f1,mean_auc
dummy,0.011251,0.017696,0.129693,0.129692,0.016212,0.125000,0.028701,0.500000
random forest,0.087122,0.022741,0.548114,0.393167,0.377682,0.392651,0.376445,0.830132
LightGBM,0.390811,0.021634,0.999658,0.417741,0.416400,0.417030,0.415064,0.818013
decision tree,0.009056,0.015928,0.405802,0.335155,0.339937,0.334472,0.312095,0.760426
xgboost,0.415126,0.020319,0.896249,0.427283,0.421808,0.426441,0.421391,0.831925


Without hyperparameter optimization, xgboost achieves the highest accuracy (0.427) among all models, with minimal overfitting (train accuracy: 0.896). However, accuracy alone does not provide the full picture for this task. When predicting AirBnB prices, **precision** is more important than recall, because high precision helps hosts set competitive prices and avoid overpricing—especially when listings should be classified in lower price bins. In this case, xgboost also has the highest precision, outperforming the second-best model (LightGBM) by 0.005.

The **F1 score** is also valuable, as it balances both precision and recall to give a more complete view of model performance. Here, xgboost edges out the next best model by 0.006, indicating it is not only precise but also reasonably good at identifying listings in the correct price bins.

Finally, the **AUC (Area Under the Curve)** metric shows how well the model ranks the correct class higher than the incorrect ones. For example, if your model gives a high-priced listing a score of 0.9 and a low-priced listing a score of 0.3, the AUC is 1.0 (perfect ranking). If the scores are reversed, the AUC drops, reflecting poor ranking. In this project, xgboost achieves an AUC of 0.83, which is 0.0009 better than the second-best model.

All in all, while accuracy is a useful baseline, precision, F1 score, and AUC are more informative for pricing tasks. Xgboost consistently performs best across these metrics, making it the most suitable model for AirBnB price classification in this analysis.

Let's try hyperparameter Optimization for our best model (xgboost) to see if we can manage a higher score

### Hyperparameter Optimization for Best Model (Random Forest)

In [38]:
# Hyperparameter space for XGBoost
param_random_xgb = {
    "xgbclassifier__n_estimators": np.arange(100, 600, 50),
    "xgbclassifier__max_depth": np.arange(3, 12, 2),
    "xgbclassifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "xgbclassifier__subsample": [0.6, 0.8, 1.0],
    "xgbclassifier__colsample_bytree": [0.6, 0.8, 1.0],
    "xgbclassifier__min_child_weight": [1, 3, 5, 7],
    "xgbclassifier__gamma": [0, 0.1, 0.3, 0.5]
}

# Pipeline with preprocessing and XGBClassifier
pipe_xgb = make_pipeline(
    preprocessing_pipeline,
    XGBClassifier(
        random_state=123,
        eval_metric="mlogloss",
        use_label_encoder=False,
        verbosity=0
    )
)

# Randomized Search for XGBClassifier
rs_xgb = RandomizedSearchCV(
    pipe_xgb,
    param_distributions=param_random_xgb,
    n_iter=50,
    cv=4,
    n_jobs=1,  # or -1 for all cores
    random_state=123,
    scoring=scoring,
    refit='accuracy'  # or refit=False if you do not need the best estimator
)

# Fit the model
rs_xgb.fit(X_train, y_train_bin)


pd.DataFrame(rs_xgb.cv_results_)


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_xgbclassifier__subsample,param_xgbclassifier__n_estimators,param_xgbclassifier__min_child_weight,param_xgbclassifier__max_depth,param_xgbclassifier__learning_rate,param_xgbclassifier__gamma,...,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro,split0_test_roc_auc_ovr,split1_test_roc_auc_ovr,split2_test_roc_auc_ovr,split3_test_roc_auc_ovr,mean_test_roc_auc_ovr,std_test_roc_auc_ovr,rank_test_roc_auc_ovr
0,1.118710,0.094816,0.024105,0.000648,0.8,350,3,5,0.05,0.0,...,0.419538,0.015475,6,0.822907,0.826517,0.820292,0.836054,0.826442,0.005973,12
1,0.301781,0.018529,0.019395,0.000613,1.0,150,1,5,0.10,0.3,...,0.425528,0.015154,3,0.826228,0.830517,0.817421,0.830416,0.826145,0.005326,13
2,0.517929,0.022428,0.021411,0.001286,0.8,250,5,9,0.10,0.5,...,0.413589,0.030419,18,0.819148,0.827211,0.822285,0.834209,0.825713,0.005685,19
3,0.296885,0.006717,0.020901,0.000648,0.6,100,7,7,0.05,0.5,...,0.401143,0.005597,42,0.826609,0.828945,0.817121,0.827949,0.825156,0.004712,21
4,0.416884,0.004400,0.020108,0.000205,0.8,350,5,9,0.20,0.5,...,0.401548,0.011077,41,0.821345,0.819737,0.813380,0.827877,0.820585,0.005157,37
5,0.184944,0.001259,0.018057,0.000318,1.0,100,1,3,0.01,0.1,...,0.373636,0.003919,50,0.811170,0.821429,0.794379,0.797394,0.806093,0.010884,50
6,0.691775,0.030696,0.021303,0.000614,1.0,350,1,9,0.05,0.3,...,0.416672,0.016395,8,0.823203,0.829516,0.815365,0.835280,0.825841,0.007405,17
7,1.815653,0.097338,0.033229,0.000684,1.0,450,5,11,0.01,0.5,...,0.415239,0.009066,13,0.827950,0.831464,0.819944,0.833881,0.828310,0.005270,6
8,0.998555,0.018086,0.026638,0.000822,0.8,500,7,5,0.10,0.1,...,0.407668,0.014401,37,0.811933,0.816780,0.806996,0.828993,0.816175,0.008169,44
9,0.405159,0.012410,0.019973,0.000743,0.6,250,1,9,0.20,0.3,...,0.411935,0.027014,23,0.824137,0.822844,0.811742,0.833976,0.823175,0.007881,29


# Conclusion

I have applied 2 different modelling strategies to predict and classify AirBnB housing prices:

- Model 1 (Regression with log(y)):
    - R^2 on log(y): 0.744 
    - R^2 on original y: 0.478
    - This model can be used to predict actual AirBnB listing price
- Model 2 (Classification with Interquartile Binning):
    - Accuracy is 0.398

For deployment, I will use Model 1 

In [42]:
import os, json
import pandas as pd
import joblib
# from sklearn.pipeline import Pipeline  # if you’re building it here

# 1) Load the exact raw DataFrame you trained on
X = pd.read_csv("data/listings.csv")

# 2) Build or load your FULL pipeline (preprocessor + regressor)
#    Example: final_model = Pipeline([("pre", preprocessing_pipeline), ("model", best_model_regression)])
final_model = best_model_regression  # <- must include the preprocessor step

# 3) Attach the raw input schema so app.py can read it
final_model.input_schema_ = list(X.columns)

# 4) Save
os.makedirs("model", exist_ok=True)
joblib.dump(final_model, "model/final_model.pkl")

# 5) (Optional) also save a JSON schema
schema = {
    "feature_order": list(X.columns),
    "dtypes": {c: str(X[c].dtype) for c in X.columns}
}
with open("model/raw_schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)
